# Modelo 6 — Árvore de Decisão para Predição de Falhas em PSDs

Este notebook tem como objetivo desenvolver um modelo de árvore de decisão capaz de estimar se uma posição individual do sistema de Portas de Plataforma (PSD) apresentará uma falha corretiva emergencial nos próximos 7 dias.

Cada PSD será identificada pela combinação de estação, via e posição da porta. Por exemplo, `FAL_V2_PSD09` representa a posição 09 da via 2 da estação FAL.

Para construir as variáveis de entrada, serão utilizados os alarmes dos componentes associados a cada PSD, principalmente:

- `PDM`: componente móvel da porta;
- `MDU`: unidade de controle associada à posição da porta.

As ordens corretivas emergenciais ZM01 que identificam uma porta específica serão utilizadas para construir o target de falha.

O desenvolvimento será dividido nas seguintes etapas:

1. Importação das bibliotecas;
2. Carregamento dos dados;
3. Análise e preparação das bases;
4. Identificação das PSDs e de seus componentes;
5. Preparação das ordens corretivas ZM01;
6. Construção das variáveis preditoras;
7. Construção do target de falha nos próximos 7 dias;
8. Separação temporal dos dados;
9. Treinamento da árvore de decisão;
10. Avaliação e interpretação dos resultados.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
# Localiza a raiz do projeto, independentemente de onde o Jupyter foi iniciado
diretorio_atual = Path.cwd().resolve()

if (diretorio_atual / "data").exists():
    raiz_projeto = diretorio_atual
elif (diretorio_atual.parent / "data").exists():
    raiz_projeto = diretorio_atual.parent
elif (diretorio_atual.parent.parent / "data").exists():
    raiz_projeto = diretorio_atual.parent.parent
else:
    raise FileNotFoundError("Não foi possível localizar a pasta data do projeto.")

caminho_alarmes = (
    raiz_projeto
    / "data"
    / "Alarmes"
    / "alarmes_trata_limpos_consolidado.csv"
)

caminho_zm01 = (
    raiz_projeto
    / "data"
    / "Corretiva Emergencial"
    / "processados"
    / "ZM01_limpa.xlsx"
)

print("Raiz do projeto:", raiz_projeto)
print("Base de alarmes encontrada:", caminho_alarmes.exists())
print("Base ZM01 encontrada:", caminho_zm01.exists())

Raiz do projeto: /home/kaian/Documentos/modulo7/g04
Base de alarmes encontrada: True
Base ZM01 encontrada: True


In [4]:
colunas_alarmes = [
    "TAG",
    "dt_alarme",
    "duracao_calculada_seg",
    "estacao_extraida",
    "plataforma_extraida",
    "porta_extraida",
    "Alarme"
]

alarmes = pd.read_csv(
    caminho_alarmes,
    usecols=colunas_alarmes,
    parse_dates=["dt_alarme"],
    low_memory=False
)

# Padroniza o nome da coluna para facilitar as próximas etapas
alarmes = alarmes.rename(
    columns={
        "Alarme": "tipo_alarme"
    }
)

print(f"Quantidade de registros: {len(alarmes):,}")
print(f"Quantidade de colunas: {alarmes.shape[1]}")

display(alarmes.head())

Quantidade de registros: 742,506
Quantidade de colunas: 7


,TAG,tipo_alarme,dt_alarme,duracao_calculada_seg,estacao_extraida,plataforma_extraida,porta_extraida
0,PSD_FAL_FAL2_PDM09_OBS,OBS,2026-05-12 08:16:52.657,1.013,FAL,FAL2,9.0
1,PSD_LUZ_LUZ1_PDM08_OBS,OBS,2026-05-12 08:15:09.913,1.004,LUZ,LUZ1,8.0
2,PSD_FAL_FAL1_PDM15_OBS,OBS,2026-05-12 08:14:06.657,1.013,FAL,FAL1,15.0
3,PSD_LUZ_LUZ1_PDM19_OBS,OBS,2026-05-12 08:09:31.930,0.987,LUZ,LUZ1,19.0
4,PSD_FRE_FRE1_PDM14_OBS,OBS,2026-05-12 08:05:05.830,9.017,FRE,FRE1,14.0


In [5]:
print("Dimensão da base:")
print(f"Linhas: {alarmes.shape[0]:,}")
print(f"Colunas: {alarmes.shape[1]}")

print("\nPeríodo dos alarmes:")
print("Data inicial:", alarmes["dt_alarme"].min())
print("Data final:", alarmes["dt_alarme"].max())

print("\nTipos das colunas:")
display(alarmes.dtypes.to_frame("tipo"))

print("\nValores ausentes:")
ausencias_alarmes = (
    alarmes.isna()
    .sum()
    .to_frame("quantidade_ausente")
)

ausencias_alarmes["percentual_ausente"] = (
    ausencias_alarmes["quantidade_ausente"]
    / len(alarmes)
    * 100
).round(2)

display(ausencias_alarmes)

Dimensão da base:
Linhas: 742,506
Colunas: 7

Período dos alarmes:
Data inicial: 2023-02-27 06:26:56.897000
Data final: 2026-05-12 08:16:52.657000

Tipos das colunas:


,tipo
TAG,str
tipo_alarme,str
dt_alarme,datetime64[us]
duracao_calculada_seg,float64
estacao_extraida,str
plataforma_extraida,str
porta_extraida,float64



Valores ausentes:


,quantidade_ausente,percentual_ausente
TAG,0,0.00
tipo_alarme,0,0.00
dt_alarme,0,0.00
duracao_calculada_seg,0,0.00
estacao_extraida,0,0.00
plataforma_extraida,0,0.00
porta_extraida,170,0.02


## 1. Identificação das PSDs e de seus componentes

Cada posição do sistema PSD será identificada pela combinação entre estação, via e número da porta. Os alarmes dos componentes `PDM` e `MDU` que possuem a mesma posição serão associados à mesma PSD.

Registros gerais do sistema que não identificam uma posição individual não serão utilizados neste modelo.

In [6]:
quantidade_inicial = len(alarmes)

# Padroniza a TAG antes da extração
tag_padronizada = (
    alarmes["TAG"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identifica os componentes individualizáveis do sistema PSD
alarmes_psd = alarmes.copy()

alarmes_psd["componente"] = tag_padronizada.str.extract(
    r"(PDM|MDU)(?=\d)",
    expand=False
)

# Extrai a via de valores como FAL1, FAL2, BUT1 e BUT2
alarmes_psd["via"] = (
    alarmes_psd["plataforma_extraida"]
    .astype("string")
    .str.extract(r"(\d+)$", expand=False)
    .astype("Int64")
)

# Padroniza a estação
alarmes_psd["estacao"] = (
    alarmes_psd["estacao_extraida"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Converte o número da posição para inteiro
alarmes_psd["posicao_psd"] = (
    pd.to_numeric(
        alarmes_psd["porta_extraida"],
        errors="coerce"
    )
    .astype("Int64")
)

# Padroniza o tipo do alarme
alarmes_psd["tipo_alarme"] = (
    alarmes_psd["tipo_alarme"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Mantém apenas registros atribuíveis a uma PSD individual
alarmes_psd = alarmes_psd[
    alarmes_psd["componente"].isin(["PDM", "MDU"])
    & alarmes_psd["via"].isin([1, 2])
    & alarmes_psd["posicao_psd"].between(1, 24)
].copy()

# Cria o identificador único da PSD
alarmes_psd["id_psd"] = (
    alarmes_psd["estacao"]
    + "_V"
    + alarmes_psd["via"].astype("string")
    + "_PSD"
    + alarmes_psd["posicao_psd"]
        .astype("string")
        .str.zfill(2)
)

# Organiza os registros cronologicamente
alarmes_psd = (
    alarmes_psd
    .sort_values(["id_psd", "dt_alarme"])
    .reset_index(drop=True)
)

print(f"Registros originais: {quantidade_inicial:,}")
print(f"Registros atribuídos às PSDs: {len(alarmes_psd):,}")
print(
    "Registros gerais removidos:",
    f"{quantidade_inicial - len(alarmes_psd):,}"
)
print(f"Estações: {alarmes_psd['estacao'].nunique()}")
print(f"PSDs identificadas: {alarmes_psd['id_psd'].nunique()}")

print("\nRegistros por componente:")
display(
    alarmes_psd["componente"]
    .value_counts()
    .rename_axis("componente")
    .to_frame("quantidade")
)

display(
    alarmes_psd[
        [
            "TAG",
            "dt_alarme",
            "id_psd",
            "componente",
            "tipo_alarme",
            "duracao_calculada_seg"
        ]
    ].head()
)

Registros originais: 742,506
Registros atribuídos às PSDs: 742,336
Registros gerais removidos: 170
Estações: 11
PSDs identificadas: 526

Registros por componente:


,quantidade
componente,
PDM,678166
MDU,64170


,TAG,dt_alarme,id_psd,componente,tipo_alarme,duracao_calculada_seg
0,PSD_BUT_BUT1_PDM01_OBS,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,OBS,2573380.513
1,PSD_P_BUT1_PDM01_CA,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,CA,2573380.513
2,PSD_P_BUT1_PDM01_CSD,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,CSD,2573380.513
3,PSD_P_BUT1_PDM01_FC,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,FC,2573380.513
4,PSD_P_BUT1_PDM01_FO,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,FO,2573380.513


## 2. Carregamento das ordens corretivas emergenciais ZM01

A base ZM01 contém as ordens de manutenção corretiva emergencial do sistema PSD. Ela será utilizada para identificar a ocorrência e o momento das falhas.

Nesta etapa, a base será apenas carregada e inspecionada. A associação entre cada ordem e uma PSD individual será realizada posteriormente por meio do local de instalação.

In [7]:
colunas_zm01 = [
    "linha_origem",
    "ordem",
    "descricao",
    "local_de_instalacao",
    "inicio_avaria_datetime",
    "falha_flag",
    "ordem_ausente_flag",
    "ordem_repetida_flag",
    "status_do_sistema_da_ordem",
    "sintoma",
    "objeto",
    "defeito",
    "causa"
]

zm01 = pd.read_excel(
    caminho_zm01,
    sheet_name="ZM01_limpa",
    usecols=colunas_zm01
)

# Garante que o início da avaria seja tratado como data e hora
zm01["inicio_avaria_datetime"] = pd.to_datetime(
    zm01["inicio_avaria_datetime"],
    errors="coerce"
)

print("Dimensão da ZM01:")
print(f"Linhas: {zm01.shape[0]:,}")
print(f"Colunas: {zm01.shape[1]}")

print("\nPeríodo das ordens:")
print(
    "Data inicial:",
    zm01["inicio_avaria_datetime"].min()
)
print(
    "Data final:",
    zm01["inicio_avaria_datetime"].max()
)

print("\nDistribuição da flag de falha:")
display(
    zm01["falha_flag"]
    .value_counts(dropna=False)
    .rename_axis("falha_flag")
    .to_frame("quantidade")
)

print("\nValores ausentes:")
ausencias_zm01 = (
    zm01.isna()
    .sum()
    .to_frame("quantidade_ausente")
)

ausencias_zm01["percentual_ausente"] = (
    ausencias_zm01["quantidade_ausente"]
    / len(zm01)
    * 100
).round(2)

display(ausencias_zm01)

display(zm01.head())

Dimensão da ZM01:
Linhas: 10,543
Colunas: 13

Período das ordens:
Data inicial: 2023-01-01 02:39:00
Data final: 2026-04-01 22:43:00

Distribuição da flag de falha:


,quantidade
falha_flag,
False,7150
True,3393



Valores ausentes:


,quantidade_ausente,percentual_ausente
linha_origem,0,0.00
ordem,1,0.01
descricao,0,0.00
local_de_instalacao,0,0.00
inicio_avaria_datetime,0,0.00
falha_flag,0,0.00
ordem_ausente_flag,0,0.00
ordem_repetida_flag,0,0.00
status_do_sistema_da_ordem,1,0.01
sintoma,0,0.00


,linha_origem,ordem,descricao,local_de_instalacao,inicio_avaria_datetime,falha_flag,ordem_ausente_flag,ordem_repetida_flag,status_do_sistema_da_ordem,sintoma,objeto,defeito,causa
0,2,53954586.0,PSD PL 01-02 Sem status para o CCO,SP04-6-PSD-BUT,2025-11-14 06:35:28,True,False,False,LIB CNPA CAPC ERRD MOME MATC NOLQ,LAÇO DE PDM ABERTO,NaN,NaN,NaN
1,3,54145839.0,PSD - PDM's não abriram PLT02,SP04-6-PSD-BUT,2026-02-27 07:33:00,False,False,False,ENTE CNPA CAPC NOLQ SCDM,TODAS AS PDM DA FACHADA NÃO ABREM,CABEAMENTO,CONTROLADOR PCC COMUNICACAO,ERRO DE COMUNICAÇÃO
2,4,52913343.0,PSD server s/ status das plataformas,SP04-6-PSD-BUT-P1S1,2024-04-18 06:00:00,False,False,False,ENTE CNPA CAPC NOLQ SCDM,PSD SERVER COM INDICAÇÃO DE FALHA NA PLA,RACK WCN,SENSOR DESTRAVADO AVARIADO,DESGASTE
3,5,53724958.0,BUT PLT 1 PORTAS NAO ABRIRAM,SP04-6-PSD-BUT-P1S1,2025-07-07 18:38:00,False,False,False,ENTE CNPA CAPC NOLQ SCDM,TODAS AS PDM DA FACHADA NÃO ABREM,CABO/CONECTORES,SWITCH TRAVADO,DESGASTE
4,6,53655728.0,PSD plt 1 - Falha geral de abertura,SP04-6-PSD-BUT-P1S1-PSSE1,2025-05-28 09:10:00,False,False,False,ENTE CNPA CAPC NOLQ SCDM,PDM NÃO ABRE,CABEAMENTO,BAIXA ISOLACAO,FALHA CBTC


## 3. Identificação das PSDs nas ordens ZM01

Para relacionar os alarmes às manutenções, será criado na ZM01 o mesmo identificador utilizado na base de alarmes.

Somente ordens cujo local de instalação informa estação, via e posição da porta podem ser atribuídas a uma PSD individual. Ordens gerais de estação, plataforma ou outros componentes serão preservadas na base original, mas não serão utilizadas como target individual.

A ocorrência de uma ordem ZM01 associada a uma PSD será utilizada como proxy de falha corretiva emergencial. A coluna `falha_flag` não será usada como filtro, pois sua ausência não significa necessariamente que não houve falha.

In [8]:
# Padroniza o local de instalação
local_padronizado = (
    zm01["local_de_instalacao"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Extrai estação, via e posição de locais como:
# SP04-6-PSD-BUT-PLT1-PP01PDM
identificacao_zm01 = local_padronizado.str.extract(
    (
        r"^SP04-6-PSD-"
        r"(?P<estacao>[A-Z]{3})-"
        r"PLT(?P<via>[12])-"
        r"PP(?P<posicao_psd>\d{2})PDM"
        r"(?:$|-)"
    )
)

# Junta a identificação extraída à base
zm01_identificada = pd.concat(
    [
        zm01.copy(),
        identificacao_zm01
    ],
    axis=1
)

# Mantém somente ordens atribuíveis a uma PSD individual
zm01_psd = zm01_identificada.dropna(
    subset=[
        "estacao",
        "via",
        "posicao_psd"
    ]
).copy()

# Converte via e posição para inteiros
zm01_psd["via"] = (
    pd.to_numeric(zm01_psd["via"], errors="coerce")
    .astype("Int64")
)

zm01_psd["posicao_psd"] = (
    pd.to_numeric(
        zm01_psd["posicao_psd"],
        errors="coerce"
    )
    .astype("Int64")
)

# Cria o mesmo identificador utilizado nos alarmes
zm01_psd["id_psd"] = (
    zm01_psd["estacao"]
    + "_V"
    + zm01_psd["via"].astype("string")
    + "_PSD"
    + zm01_psd["posicao_psd"]
        .astype("string")
        .str.zfill(2)
)

# Define o momento do evento corretivo
zm01_psd["dt_evento_corretivo"] = (
    zm01_psd["inicio_avaria_datetime"]
)

# Verifica se a PSD também existe na base de alarmes
ids_com_alarmes = set(
    alarmes_psd["id_psd"].unique()
)

zm01_psd["possui_historico_alarmes"] = (
    zm01_psd["id_psd"].isin(ids_com_alarmes)
)

# Organiza cronologicamente
zm01_psd = (
    zm01_psd
    .sort_values(["id_psd", "dt_evento_corretivo"])
    .reset_index(drop=True)
)

print(f"Ordens ZM01 totais: {len(zm01):,}")
print(
    "Ordens atribuídas a uma PSD individual:",
    f"{len(zm01_psd):,}"
)
print(
    "Ordens gerais ou de outros componentes:",
    f"{len(zm01) - len(zm01_psd):,}"
)
print(
    "PSDs diferentes presentes na ZM01:",
    zm01_psd["id_psd"].nunique()
)
print(
    "Ordens com PSD encontrada nos alarmes:",
    f"{zm01_psd['possui_historico_alarmes'].sum():,}"
)

print("\nDistribuição da flag de falha nas ordens individualizadas:")
display(
    zm01_psd["falha_flag"]
    .value_counts(dropna=False)
    .rename_axis("falha_flag")
    .to_frame("quantidade")
)

display(
    zm01_psd[
        [
            "ordem",
            "local_de_instalacao",
            "dt_evento_corretivo",
            "id_psd",
            "falha_flag",
            "possui_historico_alarmes"
        ]
    ].head()
)

Ordens ZM01 totais: 10,543
Ordens atribuídas a uma PSD individual: 9,260
Ordens gerais ou de outros componentes: 1,283
PSDs diferentes presentes na ZM01: 524
Ordens com PSD encontrada nos alarmes: 9,228

Distribuição da flag de falha nas ordens individualizadas:


,quantidade
falha_flag,
False,6220
True,3040


,ordem,local_de_instalacao,dt_evento_corretivo,id_psd,falha_flag,possui_historico_alarmes
0,52149458.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2023-02-12 10:35:00,BUT_V1_PSD01,True,True
1,52485870.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2023-08-25 16:53:00,BUT_V1_PSD01,False,True
2,52752879.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2024-01-21 12:22:00,BUT_V1_PSD01,True,True
3,53017687.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2024-06-17 05:45:00,BUT_V1_PSD01,True,True
4,53156524.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2024-09-01 16:16:00,BUT_V1_PSD01,True,True


## 4. Definição do período observável para modelagem

As bases de alarmes e de manutenções possuem períodos de cobertura diferentes entre estações e vias. Por isso, não é correto considerar automaticamente como “sem falha” um período no qual os dados não estavam disponíveis.

Para cada estação e via, será definido:

- O início da cobertura dos alarmes;
- O fim da cobertura dos alarmes;
- Uma janela histórica mínima de 30 dias para calcular as variáveis preditoras;
- Um horizonte futuro completo de 7 dias para observar o target.

A cobertura será calculada por estação e via, e não pela data do último alarme de cada porta. Uma PSD sem alarmes recentes pode estar funcionando normalmente, enquanto outras portas da mesma via confirmam que o sistema continuava sendo monitorado.

In [9]:
HORIZONTE_PREVISAO_DIAS = 7
JANELA_HISTORICO_DIAS = 30

# Calcula o período de cobertura dos alarmes por estação e via
cobertura_via = (
    alarmes_psd
    .groupby(
        ["estacao", "via"],
        as_index=False
    )
    .agg(
        inicio_cobertura_alarmes=(
            "dt_alarme",
            "min"
        ),
        fim_cobertura_alarmes=(
            "dt_alarme",
            "max"
        ),
        quantidade_alarmes=(
            "dt_alarme",
            "size"
        ),
        quantidade_psds=(
            "id_psd",
            "nunique"
        )
    )
)

# Identifica todas as PSDs observadas nos alarmes
psds_observadas = (
    alarmes_psd[
        [
            "id_psd",
            "estacao",
            "via",
            "posicao_psd"
        ]
    ]
    .drop_duplicates()
)

# Associa cada PSD à cobertura de sua estação e via
cobertura_psd = psds_observadas.merge(
    cobertura_via,
    on=["estacao", "via"],
    how="left",
    validate="many_to_one"
)

# Última data conhecida na base de corretivas
ultima_data_zm01 = (
    zm01_psd["dt_evento_corretivo"]
    .max()
    .normalize()
)

# A data de referência precisa possuir sete dias futuros completos
ultima_data_referencia = (
    ultima_data_zm01
    - pd.Timedelta(
        days=HORIZONTE_PREVISAO_DIAS
    )
)

# Reserva 30 dias iniciais para a criação das features históricas
cobertura_psd["inicio_modelagem"] = (
    cobertura_psd["inicio_cobertura_alarmes"]
    .dt.normalize()
    + pd.Timedelta(
        days=JANELA_HISTORICO_DIAS
    )
)

# O fim da modelagem não pode ultrapassar:
# 1. o fim dos alarmes daquela via;
# 2. a última data com target futuro completo.
cobertura_psd["fim_modelagem"] = (
    cobertura_psd["fim_cobertura_alarmes"]
    .dt.normalize()
    .clip(upper=ultima_data_referencia)
)

# Mantém apenas intervalos válidos
cobertura_psd = cobertura_psd[
    cobertura_psd["inicio_modelagem"]
    <= cobertura_psd["fim_modelagem"]
].copy()

# Calcula quantos dias poderão ser usados para cada PSD
cobertura_psd["quantidade_dias_modelagem"] = (
    cobertura_psd["fim_modelagem"]
    - cobertura_psd["inicio_modelagem"]
).dt.days + 1

# Mantém somente eventos de PSDs que também existem nos alarmes
ids_psd_observadas = set(
    cobertura_psd["id_psd"].unique()
)

eventos_corretivos_compativeis = zm01_psd[
    zm01_psd["id_psd"].isin(
        ids_psd_observadas
    )
].copy()

quantidade_linhas_estimadas = (
    cobertura_psd["quantidade_dias_modelagem"]
    .sum()
)

print(
    "Horizonte de previsão:",
    HORIZONTE_PREVISAO_DIAS,
    "dias"
)
print(
    "Janela histórica:",
    JANELA_HISTORICO_DIAS,
    "dias"
)
print(
    "Última data disponível na ZM01:",
    ultima_data_zm01.date()
)
print(
    "Última data possível de referência:",
    ultima_data_referencia.date()
)
print(
    "Estações e vias monitoradas:",
    len(cobertura_via)
)
print(
    "PSDs com período válido:",
    len(cobertura_psd)
)
print(
    "Eventos corretivos com PSD compatível:",
    f"{len(eventos_corretivos_compativeis):,}"
)
print(
    "Quantidade estimada de linhas diárias:",
    f"{quantidade_linhas_estimadas:,}"
)

display(
    cobertura_via[
        [
            "estacao",
            "via",
            "inicio_cobertura_alarmes",
            "fim_cobertura_alarmes",
            "quantidade_alarmes",
            "quantidade_psds"
        ]
    ]
)

Horizonte de previsão: 7 dias
Janela histórica: 30 dias
Última data disponível na ZM01: 2026-04-01
Última data possível de referência: 2026-03-25
Estações e vias monitoradas: 22
PSDs com período válido: 526
Eventos corretivos com PSD compatível: 9,228
Quantidade estimada de linhas diárias: 404,206


,estacao,via,inicio_cobertura_alarmes,fim_cobertura_alarmes,quantidade_alarmes,quantidade_psds
0,BUT,1,2023-10-17 21:44:30.900,2026-05-12 05:52:14.727,54407,24
1,BUT,2,2023-10-18 13:31:20.830,2026-05-11 08:25:56.703,37398,24
2,FAL,1,2023-10-17 22:53:09.517,2026-05-12 08:14:06.657,12183,24
3,FAL,2,2023-10-18 04:37:39.913,2026-05-12 08:16:52.657,15040,24
4,FRA,1,2023-10-18 21:14:59.083,2026-04-14 18:15:03.857,67416,24
5,FRA,2,2023-10-18 17:15:18.943,2026-04-14 19:28:40.883,91825,24
6,FRE,1,2023-02-27 06:26:56.897,2026-05-12 08:05:05.830,560,22
7,FRE,2,2023-02-27 16:58:24.710,2026-05-11 18:23:52.690,1601,24
8,HIG,1,2023-10-18 22:23:13.977,2026-05-12 07:52:23.820,37852,24
9,HIG,2,2023-10-17 22:22:55.643,2026-05-12 05:48:17.740,54088,24
